In [3]:
import pandas as pd
df = pd.read_excel(r"D:\Learning\baseline_scores.xlsx",sheet_name="hybrid_rerank")
df.iloc[7]['user_input']

'If a buyer wants to walk away from an order even though the supplier did nothing wrong, how much advance warning must it give?'

In [5]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_chroma import Chroma

In [9]:
# %pip install PyPDF
%pip install pymupdf4llm

   ---------------------------------------- 0.0/41.6 MB ? eta -:--:--
    --------------------------------------- 0.8/41.6 MB 4.2 MB/s eta 0:00:10
   - -------------------------------------- 1.8/41.6 MB 4.4 MB/s eta 0:00:10
   -- ------------------------------------- 2.9/41.6 MB 4.5 MB/s eta 0:00:09
   ---- ----------------------------------- 4.2/41.6 MB 4.9 MB/s eta 0:00:08
   ---- ----------------------------------- 4.7/41.6 MB 4.3 MB/s eta 0:00:09
   ----- ---------------------------------- 5.8/41.6 MB 4.5 MB/s eta 0:00:08
   ------ --------------------------------- 7.1/41.6 MB 4.7 MB/s eta 0:00:08
   -------- ------------------------------- 8.4/41.6 MB 4.9 MB/s eta 0:00:07
   -------- ------------------------------- 8.9/41.6 MB 4.6 MB/s eta 0:00:08
   --------- ------------------------------ 10.0/41.6 MB 4.7 MB/s eta 0:00:07
   ---------- ----------------------------- 11.3/41.6 MB 4.7 MB/s eta 0:00:07
   ----------- ---------------------------- 12.3/41.6 MB 4.8 MB/s eta 0:00:07
   

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyMuPDFLoader
loader = PyMuPDFLoader(file_path=r"D:\Learning\DocsForRAG\SDI PO TC-India.pdf")
content = loader.load()

content

c:\Users\mrinm\anaconda3\envs\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [1]:
import pymupdf4llm
md = pymupdf4llm.to_markdown(r"D:\Learning\DocsForRAG\SDI PO TC-India.pdf")

In [5]:
print(md)

# **GENERAL TERMS AND CONDITIONS OF PURCHASE** 

The relevant Schneider Electric India entity on whose behalf SDI will place the Purchase Order on the Supplier (“SE”) has chosen SDI India Business Private Limited (“SDI”) as its procurement integrator to procure services and goods, engage with Suppliers on Purchase Order basis, Purchase Order administration, remittance processing and issue escalation for all contractual and administrative matters on behalf of SE. 

SDI is authorized to, if needed, execute a Purchase Order on SE’s behalf with Supplier. Any Purchase Order issued by SDI to Supplier will be subject to these General Terms and Conditions of Purchase mentioned herein (hereinafter the “T&C”). 

SDI will not accept: (i) any changes to these T&C; (ii) any incorporation and/or reference of Supplier’s terms and conditions unless mutually agreed in Purchase Order. 

Order of Precedence: In the event of conflict or inconsistency between mutually agreed changes to the terms and condit

In [18]:
line = md.splitlines()
line


['# **GENERAL TERMS AND CONDITIONS OF PURCHASE** ',
 '',
 'The relevant Schneider Electric India entity on whose behalf SDI will place the Purchase Order on the Supplier (“SE”) has chosen SDI India Business Private Limited (“SDI”) as its procurement integrator to procure services and goods, engage with Suppliers on Purchase Order basis, Purchase Order administration, remittance processing and issue escalation for all contractual and administrative matters on behalf of SE. ',
 '',
 'SDI is authorized to, if needed, execute a Purchase Order on SE’s behalf with Supplier. Any Purchase Order issued by SDI to Supplier will be subject to these General Terms and Conditions of Purchase mentioned herein (hereinafter the “T&C”). ',
 '',
 'SDI will not accept: (i) any changes to these T&C; (ii) any incorporation and/or reference of Supplier’s terms and conditions unless mutually agreed in Purchase Order. ',
 '',
 'Order of Precedence: In the event of conflict or inconsistency between mutually agre

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "heading"), ("##", "Sections")]
)
chunks = splitter.split_text(md)   # each chunk tagged with its header in metadata

print(chunks)

[Document(metadata={'h1': '**GENERAL TERMS AND CONDITIONS OF PURCHASE**'}, page_content='The relevant Schneider Electric India entity on whose behalf SDI will place the Purchase Order on the Supplier (“SE”) has chosen SDI India Business Private Limited (“SDI”) as its procurement integrator to procure services and goods, engage with Suppliers on Purchase Order basis, Purchase Order administration, remittance processing and issue escalation for all contractual and administrative matters on behalf of SE.  \nSDI is authorized to, if needed, execute a Purchase Order on SE’s behalf with Supplier. Any Purchase Order issued by SDI to Supplier will be subject to these General Terms and Conditions of Purchase mentioned herein (hereinafter the “T&C”).  \nSDI will not accept: (i) any changes to these T&C; (ii) any incorporation and/or reference of Supplier’s terms and conditions unless mutually agreed in Purchase Order.  \nOrder of Precedence: In the event of conflict or inconsistency between mutu

In [21]:
chunks

[Document(metadata={'h1': '**GENERAL TERMS AND CONDITIONS OF PURCHASE**'}, page_content='The relevant Schneider Electric India entity on whose behalf SDI will place the Purchase Order on the Supplier (“SE”) has chosen SDI India Business Private Limited (“SDI”) as its procurement integrator to procure services and goods, engage with Suppliers on Purchase Order basis, Purchase Order administration, remittance processing and issue escalation for all contractual and administrative matters on behalf of SE.  \nSDI is authorized to, if needed, execute a Purchase Order on SE’s behalf with Supplier. Any Purchase Order issued by SDI to Supplier will be subject to these General Terms and Conditions of Purchase mentioned herein (hereinafter the “T&C”).  \nSDI will not accept: (i) any changes to these T&C; (ii) any incorporation and/or reference of Supplier’s terms and conditions unless mutually agreed in Purchase Order.  \nOrder of Precedence: In the event of conflict or inconsistency between mutu

In [9]:
# --- straight into LangChain (drop-in for your pipeline) ---
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType
from docling.chunking import HybridChunker

loader = DoclingLoader(
    file_path="D:\Learning\RAG_Pipeline\SDI PO TC-India.pdf",
    export_type=ExportType.DOC_CHUNKS,
    chunker=HybridChunker(),
)
chunks = loader.load()                          # returns LangChain Documents, ready for Chroma/BM25

[INFO] 2026-07-02 23:09:43,997 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-02 23:09:44,022 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\mrinm\anaconda3\envs\myenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-02 23:09:44,022 [RapidOCR] main.py:63: Using C:\Users\mrinm\anaconda3\envs\myenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-02 23:09:44,171 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-02 23:09:44,182 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\mrinm\anaconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-02 23:09:44,182 [RapidOCR] main.py:63: Using C:\Users\mrinm\anaconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-02 23:09:44,283 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-02 23:09:44,318 [RapidOCR] dow

In [11]:
chunks

[Document(metadata={'source': 'D:\\Learning\\RAG_Pipeline\\SDI PO TC-India.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/1', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 72.024, 't': 697.74016, 'r': 542.4195999999998, 'b': 658.89784, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 431]}]}, {'self_ref': '#/texts/2', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 72.024, 't': 645.8701599999999, 'r': 542.3927199999999, 'b': 627.69784, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 240]}]}, {'self_ref': '#/texts/3', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 72.024, 't': 614.91016, 'r': 542.1539200000005, 'b': 596.73784, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 

In [7]:
# --- native ---
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker

doc = DocumentConverter().convert(r"D:\Learning\RAG_Pipeline\SDI PO TC-India.pdf").document
chunker = HybridChunker()                      # defaults work; can pass a tokenizer
for chunk in chunker.chunk(dl_doc=doc):
    text = chunker.contextualize(chunk)
    print(f"\n\nChunk Started-{text}")        # header-enriched text to embed

[INFO] 2026-07-02 22:54:17,917 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-02 22:54:17,941 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\mrinm\anaconda3\envs\myenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-02 22:54:17,945 [RapidOCR] main.py:63: Using C:\Users\mrinm\anaconda3\envs\myenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-02 22:54:18,136 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-02 22:54:18,144 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\mrinm\anaconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-02 22:54:18,144 [RapidOCR] main.py:63: Using C:\Users\mrinm\anaconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-02 22:54:18,258 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-02 22:54:18,299 [RapidOCR] dow



Chunk Started-GENERAL TERMS AND CONDITIONS OF PURCHASE
The relevant Schneider Electric India entity on whose behalf SDI will place the Purchase Order on the Supplier ('SE') has chosen SDI India Business Private Limited ('SDI') as its procurement integrator to procure services and goods, engage with Suppliers on Purchase  Order  basis,  Purchase  Order  administration,  remittance  processing  and  issue  escalation  for  all  contractual  and administrative matters on behalf of SE.
SDI is authorized to, if needed, execute a Purchase Order on SE's behalf with Supplier. Any Purchase Order issued by SDI to Supplier will be subject to these General Terms and Conditions of Purchase mentioned herein (hereinafter the 'T&C').
SDI will not accept: (i) any changes to these T&C; (ii) any incorporation and/or reference of Supplier's terms and conditions unless mutually agreed in Purchase Order.


Chunk Started-GENERAL TERMS AND CONDITIONS OF PURCHASE
Order of Precedence: In the event of conflict

In [5]:
print(chunk)

text='8. 31.4.In  addition  to  the  above  and in  case the  performance  of  the  Purchase  Order  necessitates specific  or enhanced  protection measures for Data, the Parties will enter into a specific and appropriate addendum considering the level  of cybersecurity required by the circumstances as reasonably determined by SE and SDI.' meta=DocMeta(schema_name='docling_core.transforms.chunker.DocMeta', version='1.0.0', doc_items=[DocItem(self_ref='#/texts/212', parent=RefItem(cref='#/groups/30'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.LIST_ITEM: 'list_item'>, prov=[ProvenanceItem(page_no=10, bbox=BoundingBox(l=72.024, t=393.32016, r=542.3816799999997, b=364.82784000000004, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 331))], source=[], comments=[])], headings=['31. DATA AND CYBER SECURITY TERMS (MINIMUM REQUIREMENTS)'], captions=None, origin=DocumentOrigin(mimetype='application/pdf', binary_hash=11157499563857040

In [ ]:
# from unstructured.partition.pdf import partition_pdf
# from unstructured.chunking.title import chunk_by_title

# elements = partition_pdf(filename="D:\Learning\RAG_Pipeline\SDI PO TC-India.pdf")           # titles, paragraphs, lists, tables
# chunks = chunk_by_title(elements, max_characters=1500, new_after_n_chars=1200)
# texts = [c.text for c in chunks]

# texts

ImportError: cannot import name 'open_filename' from 'pdfminer.utils' (c:\Users\mrinm\anaconda3\envs\myenv\lib\site-packages\pdfminer\utils.py)

In [ ]:
vector_store = Chroma(collection_name='learning',embedding_function=self.embeddings,persist_directory="./Chroma_langchain_DB")
vector_store = vector_store.as_retriever(search_type = "mmr", search_kwargs = {"k":5, "lambda_mult": 0.75})

bm25_retriever = BM25Retriever.from_documents(documents=self.splitter())
bm25_retriever.k = 5


        hybrid_retriever = EnsembleRetriever(retrievers = [vector_store_retriever,bm25_retriever],weights =[0.5,0.5])

        compressor = CrossEncoderReranker(model=HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base"),top_n=3)
        compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=hybrid_retriever)


In [9]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(file_path=r"D:\Learning\DocsForRAG\SDI PO TC-India.pdf")
content = loader.load()

for d in content:
    print(d.page_content)
    print("\n\n")

1 
Schneider Electric-General Terms & Conditions of Purchase/Tail Program/ April 2021 /Ver 1.0 
GENERAL TERMS AND CONDITIONS OF PURCHASE 
The relevant Schneider Electric India entity on whose behalf SDI will place the Purchase Order on the Supplier (“SE”) has chosen 
SDI India Business Private Limited (“SDI”) as its procurement integrator to procure services and goods, engage with Suppliers on 
Purchase Order basis, Purchase Order administration, remittance processing and issue escalation for all contractual and 
administrative matters on behalf of SE.  
 
SDI is authorized to, if needed, execute a Purchase Order on SE’s behalf with Supplier. Any Purchase Order issued by SDI to 
Supplier will be subject to these General Terms and Conditions of Purchase mentioned herein (hereinafter the “T&C”). 
 
SDI will not accept: (i) any changes to these T&C; (ii) any incorporation and/or reference of Supplier’s terms and conditions unless 
mutually agreed in Purchase Order. 
Order of Precedence: I

In [19]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
db = Chroma(persist_directory=r"D:\Learning\Chroma_langchain_DB",collection_name="learning",embedding_function=HuggingFaceEmbeddings(model = "sentence-transformers/all-mpnet-base-v2"))
print(db)
result = db.get()
print(result)
print(len(result))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2748.78it/s]


{'ids': ['1d346f78-b0bd-41f8-8ba5-1e31fa66ea61', 'cff562f2-0571-4cd2-a844-f34ed661f112', '371be926-d30c-4de0-9d2e-54651445799f', '06b5c308-ee0b-4bdb-b708-794b656d3e2c', 'c672e234-63e2-4962-91c6-1a9854edf4a2', 'afe855d4-1419-4c9f-ba03-39febe5db4bb', '6ec9fb79-8520-4c00-a6d5-663dc52396df', '0c8c6ba1-c42b-4973-99a9-58a538cfbdc1', '137af765-07e6-4f19-a89f-f25608508692', '9ab18c25-602e-4eab-9bca-5fefee45bdeb', '10dbc8af-ecaa-4c16-86fc-5ee342a0872d', '826c2d78-d5df-4cc0-b089-123dfc3433ff', '53b38c6b-33c7-4a2a-8334-5b82e5a71465', 'ff7d1c97-2e4a-44a7-8709-87eeb8c0aeaf', '0635a656-20c8-4a58-afcc-7d4bd021108b', '699e9102-1564-4102-bdf3-ee77645e5394', 'a3f465bb-65c0-47a9-b447-cdd235ce1952', '11868a36-d70e-4745-b748-f4192ab63094', 'cab65c2e-a59e-4eca-a0c3-3e1e0112c853', '59db0d0e-8c74-4b21-9aea-011f6ea79dd8', 'a545fde5-4306-4cdf-9344-c9148e37c9ae', '3d868759-2064-4a5a-bc12-50cef45b6265', 'd1a85c89-32e5-4e34-89e4-0e2c1cc51edb', '1937cde7-e40e-455b-b151-3dbbbe2ea7ce', '01ebd48d-87e3-4efa-8074-abfb69

In [20]:
result

{'ids': ['1d346f78-b0bd-41f8-8ba5-1e31fa66ea61',
  'cff562f2-0571-4cd2-a844-f34ed661f112',
  '371be926-d30c-4de0-9d2e-54651445799f',
  '06b5c308-ee0b-4bdb-b708-794b656d3e2c',
  'c672e234-63e2-4962-91c6-1a9854edf4a2',
  'afe855d4-1419-4c9f-ba03-39febe5db4bb',
  '6ec9fb79-8520-4c00-a6d5-663dc52396df',
  '0c8c6ba1-c42b-4973-99a9-58a538cfbdc1',
  '137af765-07e6-4f19-a89f-f25608508692',
  '9ab18c25-602e-4eab-9bca-5fefee45bdeb',
  '10dbc8af-ecaa-4c16-86fc-5ee342a0872d',
  '826c2d78-d5df-4cc0-b089-123dfc3433ff',
  '53b38c6b-33c7-4a2a-8334-5b82e5a71465',
  'ff7d1c97-2e4a-44a7-8709-87eeb8c0aeaf',
  '0635a656-20c8-4a58-afcc-7d4bd021108b',
  '699e9102-1564-4102-bdf3-ee77645e5394',
  'a3f465bb-65c0-47a9-b447-cdd235ce1952',
  '11868a36-d70e-4745-b748-f4192ab63094',
  'cab65c2e-a59e-4eca-a0c3-3e1e0112c853',
  '59db0d0e-8c74-4b21-9aea-011f6ea79dd8',
  'a545fde5-4306-4cdf-9344-c9148e37c9ae',
  '3d868759-2064-4a5a-bc12-50cef45b6265',
  'd1a85c89-32e5-4e34-89e4-0e2c1cc51edb',
  '1937cde7-e40e-455b-b151-

In [11]:
result = db.get(include=["documents", "metadatas"])
for text, metadata in zip(result["documents"], result["metadatas"]):
    print(f"{text} & {metadata}")


In [15]:
db._collection.count() == 0

True